# IF29 — Analyse exploratoire des données (EDA)

## Projet : Détection de profils atypiques sur X (Twitter)

### Logique du projet

```
raw/*.json  →  MongoDB (tweets)  →  scripts/aggregated.sh  →  users_aggregated.csv
                                              ↓
                                    EDA (CE NOTEBOOK — 16 sections)
                                              ↓
                                    Justification : 21 → 16 features ML
                                              ↓
                                    Labellisation Excel  →  users_labeled_manual.csv
                                              ↓
                                    Modélisation ML
```

**Règle :** l'EDA porte **uniquement** sur `users_aggregated.csv` (643 124 profils · 21 variables · **sans label**).  
**Aucune lecture de tweets bruts ni de fichiers JSON** dans ce notebook — l'agrégation MongoDB est documentée dans `scripts/aggregated.sh` et `docs/RAPPORT_PROJET.md` (§ 3.2).

**Livrable :** justification des **16 features** retenues pour le ML.

### Plan du notebook (16 sections)

| § | Section |
|---|---------|
| 1 | Cadre analytique |
| 2 | Chargement et vue d'ensemble |
| 3 | Statistiques descriptives (jeu complet) |
| 4 | Qualité des données |
| 5 | Variables catégorielles et binaires |
| 6 | Distributions — 16 features ML (log1p) |
| 7 | Boxplots et percentiles |
| 8 | Relations bivariées clés |
| 9 | Sélection des variables — 21 → 16 |
| 10 | Matrice de corrélation (%) |
| 11 | Feature engineering exploratoire |
| 12 | Profils extrêmes — IQR |
| 13 | Seuils exploratoires (pré-labellisation) |
| 14 | ACP — réduction dimensionnelle |
| 15 | Normalisation — StandardScaler |
| 16 | Synthèse EDA et prochaine étape |


## 1. Cadre analytique

> *« Un profil atypique est un profil dont les caractéristiques statistiques ou comportementales s'écartent significativement de la population moyenne. »*

| Dimension | Indicateurs | Signal d'anomalie |
|-----------|-------------|-------------------|
| Graphe social | followers, friends, ratio | Déséquilibre social extrême |
| Activité | nb_tweets, retweet_ratio | Amplification / burst |
| Contenu | hashtags, URLs, mentions | Spam / surcharge |
| Temporel | active_days, tweet_frequency | Activité concentrée |
| Visibilité (SPOT) | followers × retweet_ratio | Portée d'amplification |

Références : Ferrara et al. (2016) · Chu et al. (2012) · Varol et al. (2017)

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

DATA_AGG = "users_aggregated.csv"
RANDOM_STATE = 42

VARIABLES_AGGREGATED_COUNT = 21
FEATURES_ML_COUNT = 16

FEATURES_EXCLUDED_FROM_ML = [
    "user_id", "screen_name", "profile_lang",
    "first_tweet_date", "last_tweet_date",
]

FEATURES_ML = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "nb_retweets", "retweet_ratio",
    "avg_tweet_length", "avg_hashtags", "avg_urls", "avg_mentions",
    "avg_favorites", "avg_retweet_count",
    "active_days", "tweet_frequency", "verified", "default_profile_image",
]

print("EDA — source unique :", DATA_AGG)
print(f"Agrégation : {VARIABLES_AGGREGATED_COUNT} variables → ML : {FEATURES_ML_COUNT} features")
print("Pipeline amont : scripts/aggregated.sh (voir docs/RAPPORT_PROJET.md § 3.2)")


## 2. Chargement et vue d'ensemble

In [ ]:
df = pd.read_csv(DATA_AGG)
print(f"Dataset : {df.shape[0]:,} profils × {df.shape[1]} colonnes")
print(f"Colonnes : {df.columns.tolist()}")
assert list(df.columns) == VARIABLES_ALL_21 or set(VARIABLES_ALL_21).issubset(df.columns)
df.head(10)

## 3. Statistiques descriptives (jeu complet — 643 124 profils)

In [ ]:
df_eda = df.copy()
df_eda["verified"] = df_eda["verified"].astype(int)
df_eda["default_profile_image"] = df_eda["default_profile_image"].astype(int)

num_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()
describe_full = df_eda[num_cols].describe().T.round(3)
describe_full["skew"] = df_eda[num_cols].skew().round(3)
describe_full["kurtosis"] = df_eda[num_cols].kurtosis().round(3)

print("=" * 70)
print(f"STATISTIQUES DESCRIPTIVES — {len(df):,} profils")
print("=" * 70)
display(describe_full)

print(f"\n{len(num_cols)} colonnes numériques sur {df.shape[1]} colonnes agrégées.")

## 4. Qualité des données

In [ ]:
print("=" * 60)
print("TYPES, DOUBLONS ET VALEURS MANQUANTES")
print("=" * 60)
df.info()

n_dup_id = df["user_id"].duplicated().sum()
n_dup_name = df["screen_name"].duplicated().sum()
print(f"\nDoublons user_id : {n_dup_id}")
print(f"Doublons screen_name : {n_dup_name}")

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(3)
missing_df = pd.DataFrame({
    "Manquantes": missing,
    "Pourcentage (%)": missing_pct,
}).sort_values("Pourcentage (%)", ascending=False)
display(missing_df)

missing_pos = missing_df[missing_df["Manquantes"] > 0]
if missing_pos.empty:
    print("\nAucune valeur manquante — jeu directement exploitable.")
else:
    missing_pos["Pourcentage (%)"].plot(kind="barh", color="coral", figsize=(8, 4))
    plt.title("Valeurs manquantes par colonne (%)")
    plt.xlabel("%")
    plt.tight_layout()
    plt.show()

## 5. Variables catégorielles et binaires

Analyse de `verified`, `default_profile_image` et `profile_lang` avant exclusion du ML.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
    ["verified", "default_profile_image"],
    ["Compte vérifié", "Avatar par défaut"]):
    vc = df[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values, color=["#1565C0", "#C62828"][:len(vc)], edgecolor="black")
    ax.set_title(title)
    for i, v in enumerate(vc.values):
        ax.text(i, v, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=9)

top_lang = df["profile_lang"].value_counts().head(10)
axes[2].barh(top_lang.index[::-1], top_lang.values[::-1], color="teal", edgecolor="black")
axes[2].set_title("Top 10 langues (profile_lang)")
plt.suptitle("Variables catégorielles", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Langues distinctes : {df['profile_lang'].nunique()}")

## 6. Distributions — 16 features ML (échelle log1p)

Les distributions sont fortement asymétriques (queues longues), typiques des réseaux sociaux.
`log1p` sert **uniquement à visualiser** ; le ML utilisera `StandardScaler`.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(FEATURES_ML):
    data = np.log1p(df_eda[col].clip(lower=0).astype(float))
    axes[i].hist(data, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
    med = df_eda[col].median()
    axes[i].axvline(np.log1p(med), color="red", linestyle="--", linewidth=1, label=f"méd={med:.1f}")
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].legend(fontsize=7)

plt.suptitle("Distributions des 16 features ML — log(1+x)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 7. Boxplots et percentiles (détection visuelle d'outliers)

In [ ]:
box_cols = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "retweet_ratio", "avg_hashtags", "avg_urls", "active_days",
]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for ax, col in zip(axes, box_cols):
    data = df[col].clip(upper=df[col].quantile(0.95))
    ax.boxplot(data, vert=True)
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
plt.suptitle("Boxplots tronqués au 95e percentile", fontweight="bold")
plt.tight_layout()
plt.show()

percentiles = [1, 5, 25, 50, 75, 95, 99]
pct_df = df_eda[FEATURES_ML].quantile([p/100 for p in percentiles]).T
pct_df.columns = [f"P{p}" for p in percentiles]
display(pct_df.round(3))

## 8. Relations bivariées clés

In [ ]:
sample = df_eda.sample(min(20_000, len(df_eda)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

axes[0, 0].scatter(
    np.log1p(sample["followers_count"]),
    np.log1p(sample["nb_tweets"]),
    alpha=0.15, s=8, c="steelblue",
)
axes[0, 0].set_xlabel("log(1+followers)")
axes[0, 0].set_ylabel("log(1+nb_tweets)")
axes[0, 0].set_title("Followers vs Activité")

axes[0, 1].scatter(
    sample["retweet_ratio"].clip(0, 1),
    sample["avg_urls"].clip(0, sample["avg_urls"].quantile(0.99)),
    alpha=0.15, s=8, c="teal",
)
axes[0, 1].set_xlabel("retweet_ratio")
axes[0, 1].set_ylabel("avg_urls")
axes[0, 1].set_title("Amplification vs URLs")

axes[1, 0].scatter(
    np.log1p(sample["friends_count"]),
    np.log1p(sample["followers_count"]),
    alpha=0.15, s=8, c="coral",
)
axes[1, 0].set_xlabel("log(1+friends)")
axes[1, 0].set_ylabel("log(1+followers)")
axes[1, 0].set_title("Graphe social")

axes[1, 1].hist(sample["followers_friends_ratio"].clip(0, sample["followers_friends_ratio"].quantile(0.99)),
                bins=50, color="purple", alpha=0.7, edgecolor="white")
axes[1, 1].axvline(30, color="red", linestyle="--", label="Seuil labellisation (30)")
axes[1, 1].set_xlabel("followers_friends_ratio")
axes[1, 1].set_title("Ratio followers/friends")
axes[1, 1].legend()

plt.suptitle("Relations bivariées (échantillon 20 000)", fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Sélection des variables — 21 → 16 features ML

**Livrable principal.** Cinq variables exclues après analyse :

In [ ]:
selection_df = pd.DataFrame({
    "Variable exclue": FEATURES_EXCLUDED_FROM_ML,
    "Type": ["Identifiant", "Identifiant", "Catégorielle", "Temporelle", "Temporelle"],
    "Raison": [
        "Non prédictive — clé technique",
        "Non prédictive — identifiant texte",
        "Peu informative (langue) pour la détection d'anomalies",
        "Redondante avec active_days / tweet_frequency",
        "Redondante avec active_days / tweet_frequency",
    ],
})
display(selection_df)

print(f"\nVariables agrégées : {VARIABLES_AGGREGATED_COUNT}")
print(f"Features ML retenues : {FEATURES_ML_COUNT}")
print("Liste :", FEATURES_ML)
assert len(FEATURES_ML) == FEATURES_ML_COUNT
assert set(FEATURES_ML).issubset(set(df.columns))

## 10. Matrice de corrélation (%) — 16 features ML

In [ ]:
df_ml = df_eda[FEATURES_ML].astype(float)
corr_matrix = df_ml.corr()
corr_pct = (corr_matrix * 100).round(1)

plt.figure(figsize=(14, 11))
annot = corr_pct.map(lambda x: f"{x:.0f}%")
sns.heatmap(
    corr_pct, annot=annot, fmt="",
    cmap="RdBu_r", center=0, vmin=-100, vmax=100,
    linewidths=0.4, square=True,
    cbar_kws={"label": "Corrélation (%)"},
    annot_kws={"size": 7},
)
plt.title("Matrice de corrélation Pearson (%) — 16 features ML", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Paires fortement corrélées (|r| > 50 %) :")
pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_pct.iloc[i, j]
        if abs(r) > 50:
            pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], r))
            print(f"  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]} : {r:.1f} %")
if not pairs:
    print("  (aucune paire > 50 %)")

## 11. Feature engineering exploratoire

Variables dérivées inspirées de la littérature (Chu, Ferrara, SPOT).  
**Non retenues dans les 16 features ML** — exploration uniquement.

In [ ]:
df_feat = df_eda.copy()
df_feat["activity_rate"] = df_feat["nb_tweets"] / (df_feat["active_days"] + 1)
df_feat["visibility_score"] = np.log1p(df_feat["followers_count"]) * (1 + df_feat["retweet_ratio"])
df_feat["content_richness"] = df_feat["avg_hashtags"] + df_feat["avg_urls"] + df_feat["avg_mentions"]
df_feat["aggressiveness_score"] = df_feat["activity_rate"] * np.log1p(df_feat["friends_count"] + 1)

engineered = ["activity_rate", "visibility_score", "content_richness", "aggressiveness_score"]
eng_stats = df_feat[engineered].describe().T.round(3)
display(eng_stats)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), engineered):
    d = df_feat[col].replace([np.inf, -np.inf], np.nan).dropna()
    d = d.clip(d.quantile(0.01), d.quantile(0.99))
    ax.hist(d, bins=50, color="teal", edgecolor="white", alpha=0.85)
    ax.set_title(col)
plt.suptitle("Features engineered (tronquées P1–P99)", fontweight="bold")
plt.tight_layout()
plt.show()

## 12. Profils extrêmes — méthode IQR

In [ ]:
def count_outliers_iqr(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return int(((series < lower) | (series > upper)).sum())

outlier_cols = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "retweet_ratio", "avg_urls", "avg_mentions", "active_days",
]
outlier_stats = pd.DataFrame({
    "Outliers IQR": [count_outliers_iqr(df_eda[c]) for c in outlier_cols],
    "% profils": [count_outliers_iqr(df_eda[c]) / len(df) * 100 for c in outlier_cols],
    "P99": [df_eda[c].quantile(0.99) for c in outlier_cols],
    "Médiane": [df_eda[c].median() for c in outlier_cols],
}, index=outlier_cols).round(3)
display(outlier_stats)

print("\nTop 10 — ratio followers/friends :")
display(df.nlargest(10, "followers_friends_ratio")[
    ["screen_name", "followers_count", "friends_count", "followers_friends_ratio",
     "nb_tweets", "retweet_ratio", "avg_urls"]
])

print("\nTop 10 — retweet_ratio :")
display(df.nlargest(10, "retweet_ratio")[
    ["screen_name", "retweet_ratio", "nb_tweets", "nb_retweets", "avg_mentions"]
])

## 13. Seuils exploratoires (pré-labellisation)

Comptage des profils dépassant des seuils qui inspireront la labellisation Excel (voir `docs/LABELISATION.md`).

In [ ]:
thresholds = {
    "retweet_ratio ≥ 0.8": (df_eda["retweet_ratio"] >= 0.8).sum(),
    "avg_urls ≥ 1.5": (df_eda["avg_urls"] >= 1.5).sum(),
    "avg_mentions ≥ 2": (df_eda["avg_mentions"] >= 2).sum(),
    "active_days=1 & nb_tweets≥2": ((df_eda["active_days"] == 1) & (df_eda["nb_tweets"] >= 2)).sum(),
    "followers_friends_ratio ≥ 30": (df_eda["followers_friends_ratio"] >= 30).sum(),
}
thresh_df = pd.DataFrame({
    "Critère": list(thresholds.keys()),
    "Profils": list(thresholds.values()),
    "%": [v / len(df) * 100 for v in thresholds.values()],
}).round({"Profils": 0, "%": 2})
display(thresh_df)

print("\nCes seuils seront formalisés en labellisation Excel (≥ 2 critères sur 4 → atypique).")

## 14. ACP — réduction dimensionnelle (16 features, sans labels)

ACP sur `users_aggregated.csv` **avant labellisation**.  
Choix retenu pour le non supervisé : **7 composantes** (seuil 75 % de variance cumulée).

In [ ]:
X_scaled = StandardScaler().fit_transform(df_eda[FEATURES_ML].astype(float))
pca = PCA(n_components=min(16, X_scaled.shape[1]), random_state=RANDOM_STATE)
pca.fit(X_scaled)
var_ratio = pca.explained_variance_ratio_ * 100
var_cum = np.cumsum(var_ratio)

n75 = int(np.searchsorted(var_cum, 75) + 1)
n_kaiser = int((pca.explained_variance_ > 1).sum()) if len(pca.explained_variance_) == 16 else 5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(var_ratio) + 1), var_ratio, color="steelblue", edgecolor="black", label="Par composante")
axes[0].set_xlabel("Composante")
axes[0].set_ylabel("Variance expliquée (%)")
axes[0].set_title("Variance par composante (scree)")
axes[0].legend()

axes[1].plot(range(1, len(var_cum) + 1), var_cum, "o-", color="steelblue", linewidth=2)
for thresh, color, lbl in [(70, "gray", "70%"), (75, "red", "75% (retenu)"), (80, "green", "80%")]:
    axes[1].axhline(thresh, color=color, linestyle="--", alpha=0.7, label=lbl)
axes[1].axvline(n75, color="orange", linestyle=":", linewidth=2, label=f"Retenu = {n75} comp.")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].set_title("Variance cumulée")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle("ACP — 16 features ML (StandardScaler)", fontweight="bold")
plt.tight_layout()
plt.show()

print("Tableau décision ACP :")
for k in range(1, min(9, len(var_cum) + 1)):
    print(f"  {k} comp. → {var_cum[k-1]:.1f} % cumulé")
print(f"\n=> Retenu : {n75} composantes ({var_cum[n75-1]:.1f} %) pour le non supervisé")

## 15. Normalisation — vérification StandardScaler

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(df_eda[FEATURES_ML].astype(float))
norm_check = pd.DataFrame(X_norm, columns=FEATURES_ML).describe().T[["mean", "std"]].round(4)
display(norm_check.head(8))
print("\nMoyenne ≈ 0 et écart-type ≈ 1 après StandardScaler — cohérent avec le pipeline ML.")

## 16. Synthèse EDA et prochaine étape

| Constat | Détail |
|---------|--------|
| Volume | 643 124 profils agrégés |
| Qualité | Peu ou pas de valeurs manquantes |
| Forme | Distributions asymétriques (queues longues) |
| Outliers | Présents sur followers, ratio, retweet_ratio |
| Corrélations | Redondance partielle → ACP en modélisation |
| Features ML | **21 → 16** (5 exclues) |
| ACP | **7 composantes** ~ 79 % variance (non supervisé) |

**Étape suivante :** labellisation Excel → `users_labeled_manual.csv` (`Groupe3_Labelisation.ipynb`)

In [ ]:
recap_eda = pd.DataFrame({
    "Métrique": [
        "Fichier source",
        "Profils",
        "Variables agrégées",
        "Features ML retenues",
        "Comptes vérifiés",
        "Médiane followers",
        "Médiane nb_tweets",
        "Médiane retweet_ratio",
        "Composantes ACP (75%)",
        "Variance cumulée (7 comp.)",
    ],
    "Valeur": [
        DATA_AGG,
        f"{len(df):,}",
        str(VARIABLES_AGGREGATED_COUNT),
        str(FEATURES_ML_COUNT),
        f"{df['verified'].mean()*100:.2f} %",
        f"{df['followers_count'].median():,.0f}",
        f"{df['nb_tweets'].median():,.0f}",
        f"{df['retweet_ratio'].median():.3f}",
        str(n75),
        f"{var_cum[n75-1]:.1f} %",
    ],
})
print("=" * 55)
print(" RÉCAPITULATIF EDA")
print("=" * 55)
display(recap_eda)

recap_features = pd.DataFrame({"Feature ML retenue": FEATURES_ML})
display(recap_features)

print("\n→ Fin EDA. Prochaine étape : labellisation Excel.")